In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

In [2]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [3]:
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("The device we are using is ", device)

The device we are using is  cuda


In [5]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y), in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # if batch % 100 == 0:
        #     loss, current = loss.item(), batch * len(X)
            # print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()

    test_loss, correct = 0,0

    with torch.inference_mode():
        for batch, (X, y), in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            prediction = model(X)
            loss = loss_fn(prediction, y).item()

            test_loss += loss
            correct += (prediction.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test error --> Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

    return test_loss

In [9]:
base_model = NeuralNetwork().to(device)
base_model

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [13]:
cnn_model = CNN().to(device)
cnn_model

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout2d(p=0.3, inplace=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.3, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6272, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)

In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=0.001)
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.977)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',      # Looking for minimum loss
    factor=0.5,      # Halve the LR
    patience=5,      # Wait 3 epochs before reducing
)

best_loss = float('inf')
patience = 20
patience_counter = 0

epochs = 100
for t in range(epochs):
    print(f"Epoch {t + 1} (LR: {scheduler.get_last_lr()[0]:.6f})")
    print("------------------------------")
    train(train_dataloader, cnn_model, loss_fn, optimizer)

    val_loss = test(test_dataloader, cnn_model, loss_fn)
    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(cnn_model.state_dict(), "fmnist.pth")
        print(f"  Saved new best model (loss: {best_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {t+1}")
            break
    scheduler.step(val_loss)

print("Done!")

Epoch 1 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 86.2%, Avg loss: 0.364286 

  Saved new best model (loss: 0.364286)
Epoch 2 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 89.3%, Avg loss: 0.283135 

  Saved new best model (loss: 0.283135)
Epoch 3 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 90.0%, Avg loss: 0.264776 

  Saved new best model (loss: 0.264776)
Epoch 4 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 90.4%, Avg loss: 0.256355 

  Saved new best model (loss: 0.256355)
Epoch 5 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 91.0%, Avg loss: 0.241277 

  Saved new best model (loss: 0.241277)
Epoch 6 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 91.3%, Avg loss: 0.231806 

  Saved new best model (loss: 0.231806)
Epoch 7 (LR: 0.001000)
------------------------------
Test error --> Accuracy: 91.2%, Avg loss: 0.233011 

Epoch 8 (LR